# Лабораторная работа 1.3. Метод простых итераций и метод Зейделя

**Задание.** Реализовать метод простых итераций и метод Зейделя в виде программ, задавая в качестве
входных данных матрицу системы, вектор правых частей и точность вычислений. Используя разработанное ПО,
решить СЛАУ. Проанализировать количество итераций, необходимое для достижения заданной точности.

$$
\begin{cases}
18x_1 + 8x_2 - 3x_3 + 4x_4 = -84 \\
-7x_1 + 15x_2 - 5x_3 - 2x_4 = -5 \\
-4x_1 + 13x_3 + 4x_4 = -38 \\
-8x_1 - 8x_2 - 6x_3 + 31x_4 = 263
\end{cases}
$$

## Теория

**Приведение к эквивалентному виду.** Из $i$-го уравнения выражаем $x_i$:

$$x = \beta + \alpha x, \qquad
\alpha_{ij} = -\frac{a_{ij}}{a_{ii}}\ (i \ne j),\quad \alpha_{ii} = 0, \qquad \beta_i = \frac{b_i}{a_{ii}}.$$

**Метод простых итераций (Якоби):** $x^{(k+1)} = \beta + \alpha x^{(k)}$, начальное приближение $x^{(0)} = \beta$.

**Метод Зейделя:** при вычислении $x_i^{(k+1)}$ сразу используются уже найденные компоненты новой итерации:

$$x_i^{(k+1)} = \beta_i + \sum_{j<i} \alpha_{ij} x_j^{(k+1)} + \sum_{j>i} \alpha_{ij} x_j^{(k)}.$$

В матричной форме $\alpha = B + C$ ($B$ — нижняя треугольная часть, $C$ — верхняя с диагональю), тогда
$x^{(k+1)} = (E - B)^{-1} C\, x^{(k)} + (E - B)^{-1}\beta$.

**Достаточное условие сходимости** — $\|\alpha\| < 1$. Оно выполняется, например, при строгом
диагональном преобладании $|a_{ii}| > \sum_{j \ne i} |a_{ij}|$. Используется норма $\|\cdot\|_\infty$
(максимум суммы модулей по строкам).

**Критерий окончания** (апостериорная оценка погрешности):

- простые итерации: $\varepsilon^{(k)} = \dfrac{\|\alpha\|}{1 - \|\alpha\|}\,\|x^{(k)} - x^{(k-1)}\| \le \varepsilon$;
- Зейдель: $\varepsilon^{(k)} = \dfrac{\|C\|}{1 - \|\alpha\|}\,\|x^{(k)} - x^{(k-1)}\| \le \varepsilon$.

Если $\|\alpha\| \ge 1$, используется просто $\|x^{(k)} - x^{(k-1)}\| \le \varepsilon$.

**Априорная оценка числа итераций** метода простых итераций:

$$k + 1 \ge \frac{\ln \varepsilon - \ln \|\beta\| + \ln(1 - \|\alpha\|)}{\ln \|\alpha\|}.$$

## Реализация

In [1]:
import math


def norm_vec(v):
    return max(abs(x) for x in v)


def norm_mat(M):
    return max(sum(abs(x) for x in row) for row in M)


def to_iterative_form(A, b):
    """Приведение Ax = b к виду x = beta + alpha x."""
    n = len(A)
    for i in range(n):
        if A[i][i] == 0:
            raise ValueError(f"Нулевой диагональный элемент a[{i}][{i}] — переставьте уравнения")
    alpha = [[0.0 if i == j else -A[i][j] / A[i][i] for j in range(n)] for i in range(n)]
    beta = [b[i] / A[i][i] for i in range(n)]
    return alpha, beta


def simple_iterations(A, b, eps, max_iter=10_000):
    """Метод простых итераций. Возвращает (x, число итераций, история оценок погрешности)."""
    alpha, beta = to_iterative_form(A, b)
    n = len(beta)
    a_norm = norm_mat(alpha)
    coef = a_norm / (1 - a_norm) if a_norm < 1 else 1.0

    x = beta[:]
    history = []
    for k in range(1, max_iter + 1):
        x_new = [beta[i] + sum(alpha[i][j] * x[j] for j in range(n)) for i in range(n)]
        err = coef * norm_vec([x_new[i] - x[i] for i in range(n)])
        history.append(err)
        x = x_new
        if err <= eps:
            return x, k, history
    raise RuntimeError("Метод простых итераций не сошёлся за max_iter итераций")


def seidel(A, b, eps, max_iter=10_000):
    """Метод Зейделя. Возвращает (x, число итераций, история оценок погрешности)."""
    alpha, beta = to_iterative_form(A, b)
    n = len(beta)
    a_norm = norm_mat(alpha)
    c_norm = norm_mat([[alpha[i][j] if j >= i else 0.0 for j in range(n)] for i in range(n)])
    coef = c_norm / (1 - a_norm) if a_norm < 1 else 1.0

    x = beta[:]
    history = []
    for k in range(1, max_iter + 1):
        x_prev = x[:]
        for i in range(n):  # x[j] при j < i уже обновлены на этой итерации
            x[i] = beta[i] + sum(alpha[i][j] * x[j] for j in range(n))
        err = coef * norm_vec([x[i] - x_prev[i] for i in range(n)])
        history.append(err)
        if err <= eps:
            return x, k, history
    raise RuntimeError("Метод Зейделя не сошёлся за max_iter итераций")


def apriori_iterations(alpha, beta, eps):
    """Априорная оценка числа итераций метода простых итераций."""
    a, bt = norm_mat(alpha), norm_vec(beta)
    return math.ceil((math.log(eps) - math.log(bt) + math.log(1 - a)) / math.log(a) - 1)


def residual(A, b, x):
    return norm_vec([sum(A[i][j] * x[j] for j in range(len(x))) - b[i] for i in range(len(x))])

## Исходные данные

In [2]:
A = [
    [18,  8, -3,  4],
    [-7, 15, -5, -2],
    [-4,  0, 13,  4],
    [-8, -8, -6, 31],
]
b = [-84, -5, -38, 263]
eps = 1e-6

for row, bi in zip(A, b):
    print("  [" + " ".join(f"{v:4}" for v in row) + f" ]   | {bi:5}")

  [  18    8   -3    4 ]   |   -84
  [  -7   15   -5   -2 ]   |    -5
  [  -4    0   13    4 ]   |   -38
  [  -8   -8   -6   31 ]   |   263


## Проверка условия сходимости

In [3]:
print("Диагональное преобладание:")
for i, row in enumerate(A):
    off = sum(abs(v) for j, v in enumerate(row) if j != i)
    print(f"  строка {i + 1}: |a_ii| = {abs(row[i]):2}  >  {off:2} = сумма остальных  ->", abs(row[i]) > off)

alpha, beta = to_iterative_form(A, b)
print("\nalpha =")
for row in alpha:
    print("  [" + " ".join(f"{v:9.5f}" for v in row) + " ]")
print("beta  =", [round(v, 5) for v in beta])
print(f"\n||alpha||_inf = {norm_mat(alpha):.5f} < 1  ->  оба метода сходятся")

Диагональное преобладание:
  строка 1: |a_ii| = 18  >  15 = сумма остальных  -> True
  строка 2: |a_ii| = 15  >  14 = сумма остальных  -> True
  строка 3: |a_ii| = 13  >   8 = сумма остальных  -> True
  строка 4: |a_ii| = 31  >  22 = сумма остальных  -> True

alpha =
  [  0.00000  -0.44444   0.16667  -0.22222 ]
  [  0.46667   0.00000   0.33333   0.13333 ]
  [  0.30769   0.00000   0.00000  -0.30769 ]
  [  0.25806   0.25806   0.19355   0.00000 ]
beta  = [-4.66667, -0.33333, -2.92308, 8.48387]

||alpha||_inf = 0.93333 < 1  ->  оба метода сходятся


## Решение методом простых итераций

In [4]:
x_si, k_si, hist_si = simple_iterations(A, b, eps)
for i, xi in enumerate(x_si, 1):
    print(f"x{i} = {xi:.6f}")
print(f"\nИтераций: {k_si}   (априорная оценка: {apriori_iterations(alpha, beta, eps)})")
print(f"Невязка ||Ax - b|| = {residual(A, b, x_si):.2e}")

x1 = -5.000000
x2 = -4.000000
x3 = -6.000000
x4 = 5.000000

Итераций: 38   (априорная оценка: 270)
Невязка ||Ax - b|| = 5.70e-07


## Решение методом Зейделя

In [5]:
x_sd, k_sd, hist_sd = seidel(A, b, eps)
for i, xi in enumerate(x_sd, 1):
    print(f"x{i} = {xi:.6f}")
print(f"\nИтераций: {k_sd}")
print(f"Невязка ||Ax - b|| = {residual(A, b, x_sd):.2e}")

x1 = -5.000000
x2 = -4.000000
x3 = -6.000000
x4 = 5.000000

Итераций: 21
Невязка ||Ax - b|| = 3.02e-07


## Анализ количества итераций

In [6]:
print(f"{'eps':>8} | {'апр. оценка':>11} | {'простые итер.':>13} | {'Зейдель':>7}")
print("-" * 50)
for p in range(1, 11):
    e = 10.0 ** -p
    _, k1, _ = simple_iterations(A, b, e)
    _, k2, _ = seidel(A, b, e)
    print(f"{e:8.0e} | {apriori_iterations(alpha, beta, e):11} | {k1:13} | {k2:7}")

     eps | апр. оценка | простые итер. | Зейдель
--------------------------------------------------
   1e-01 |         103 |            14 |       9
   1e-02 |         136 |            19 |      11
   1e-03 |         170 |            23 |      14
   1e-04 |         203 |            28 |      16
   1e-05 |         237 |            33 |      19
   1e-06 |         270 |            38 |      21
   1e-07 |         303 |            43 |      24
   1e-08 |         337 |            47 |      26
   1e-09 |         370 |            52 |      29
   1e-10 |         403 |            57 |      31


In [7]:
print("Оценка погрешности по итерациям (eps = 1e-6):\n")
print(f"{'k':>3} | {'простые итер.':>13} | {'Зейдель':>10}")
print("-" * 33)
for k in range(max(len(hist_si), len(hist_sd))):
    s1 = f"{hist_si[k]:13.3e}" if k < len(hist_si) else " " * 13
    s2 = f"{hist_sd[k]:10.3e}" if k < len(hist_sd) else ""
    print(f"{k + 1:3} | {s1} | {s2}")

Оценка погрешности по итерациям (eps = 1e-6):

  k | простые итер. |    Зейдель
---------------------------------
  1 |     5.665e+01 |  5.913e+01
  2 |     3.688e+01 |  2.164e+01
  3 |     2.197e+01 |  1.274e+01
  4 |     1.286e+01 |  6.763e+00
  5 |     5.959e+00 |  1.406e+00
  6 |     4.066e+00 |  8.043e-01
  7 |     3.161e+00 |  3.978e-01
  8 |     1.743e+00 |  1.021e-01
  9 |     9.936e-01 |  5.724e-02
 10 |     5.949e-01 |  2.252e-02
 11 |     3.857e-01 |  7.045e-03
 12 |     2.736e-01 |  3.889e-03
 13 |     1.411e-01 |  1.217e-03
 14 |     7.376e-02 |  4.668e-04
 15 |     5.809e-02 |  2.543e-04
 16 |     3.490e-02 |  6.179e-05
 17 |     2.254e-02 |  2.985e-05
 18 |     1.078e-02 |  1.606e-05
 19 |     6.651e-03 |  3.149e-06
 20 |     5.354e-03 |  1.846e-06
 21 |     3.005e-03 |  9.815e-07
 22 |     1.759e-03 | 
 23 |     9.733e-04 | 
 24 |     6.454e-04 | 
 25 |     4.692e-04 | 
 26 |     2.464e-04 | 
 27 |     1.285e-04 | 
 28 |     9.669e-05 | 
 29 |     5.917e-05 | 
 30 |    

## Проверка с помощью NumPy (необязательно)

In [8]:
try:
    import numpy as np
    x_np = np.linalg.solve(np.array(A, float), np.array(b, float))
    print("x (numpy):", x_np)
    print("max |x_si - x_numpy| =", np.abs(np.array(x_si) - x_np).max())
    print("max |x_sd - x_numpy| =", np.abs(np.array(x_sd) - x_np).max())
except ImportError:
    print("NumPy не установлен — пропускаем сверку")

x (numpy): [-5. -4. -6.  5.]
max |x_si - x_numpy| = 3.0723144917033096e-08
max |x_sd - x_numpy| = 1.3993453507055165e-08


## Вывод

Матрица системы имеет строгое диагональное преобладание, поэтому $\|\alpha\|_\infty < 1$ и оба метода сходятся
при любом начальном приближении.

- Число итераций растёт примерно линейно с ростом $\lg(1/\varepsilon)$: каждый новый верный знак
  требует фиксированного числа дополнительных итераций: около 4–5 для простых итераций и 2–3 для Зейделя.
  Это отражает линейную (геометрическую) скорость сходимости.
- Метод Зейделя сходится примерно вдвое быстрее метода простых итераций (при $\varepsilon = 10^{-6}$:
  21 итерация против 38), так как сразу использует уточнённые компоненты текущего приближения.
- Норма $\|\alpha\|_\infty$ близка к единице, поэтому коэффициент $\frac{\|\alpha\|}{1-\|\alpha\|}$ велик,
  а априорная оценка завышает число итераций примерно в 7 раз (270 против 38 при $\varepsilon = 10^{-6}$). Фактически метод сходится быстрее,
  потому что $\|\alpha\|$ — лишь верхняя граница коэффициента сжатия.